In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'irlib.py': '"""IR of one quantitative environmental figure + deterministic comparison (GreenScan AG-1, GS-Bench v1).\n\nA model may fill the IR; every number, unit, period, scope and boundary check is done here, so the\nverdict always points at an arithmetic step (invariants 4-6 of ARCHITECTURE_INVARIANTS.md):\n  - one dimension differs (metric, value kind, status, period, scope, boundary, unit) -> N, never C;\n  - a target never proves a result;\n  - values match within the precision each side was published to.\nPure stdlib so the same file runs on Kaggle and in the product.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import asdict, dataclass, field\n\nMETRICS = {\n    "ghg_emissions": "phát thải khí nhà kính",\n    "energy_consumption": "tiêu thụ năng lượng",\n    "electricity_consumption": "tiêu thụ điện",\n    "renewable_energy": "năng lượng tái tạo (sản xuất hoặc sử dụng)",\n    "fuel_consumption": "tiêu thụ nhiên liệu (than, dầu, khí, sinh khối)",\n    "water_use": "nước khai thác / sử dụng / tiêu thụ",\n    "wastewater": "nước thải phát sinh / xử lý / tuần hoàn",\n    "waste_generated": "chất thải phát sinh",\n    "waste_recycled": "chất thải tái chế / tái sử dụng / thu hồi",\n    "hazardous_waste": "chất thải nguy hại",\n    "recycled_input": "nguyên liệu tái chế đưa vào sản xuất",\n    "air_pollutants": "bụi, SO2, NOx, khí thải khác",\n    "land_trees": "trồng cây, diện tích xanh, rừng",\n    "other_env": "chỉ số môi trường khác",\n}\nVALUE_KINDS = ("absolute", "intensity", "share", "change", "count")\nSTATUSES = ("actual", "target")\nSCOPES = ("s1", "s2", "s3", "s1+s2", "s1+s2+s3")\nBOUNDARIES = ("group", "parent", "facility", "unspecified")\n\nMULTIPLIERS = {"nghìn": 1e3, "ngàn": 1e3, "ngàn tỷ": 1e12, "nghìn tỷ": 1e12, "triệu": 1e6, "tỷ": 1e9, "tỉ": 1e9,\n               "thousand": 1e3, "million": 1e6, "billion": 1e9, "": 1.0}\n\n# unit text -> (dimension, factor to the dimension\'s base unit)\n_UNITS = [\n    (r"(kt|nghìn tấn) ?co2", ("ghg_t", 1e3)), (r"kg ?co2", ("ghg_t", 1e-3)), (r"(t|tấn) ?co2", ("ghg_t", 1.0)),\n    (r"twh", ("energy_kwh", 1e9)), (r"gwh", ("energy_kwh", 1e6)), (r"mwh", ("energy_kwh", 1e3)), (r"kwh", ("energy_kwh", 1.0)),\n    (r"toe", ("energy_kwh", 11_630.0)), (r"\\btj\\b", ("energy_kwh", 277_777.8)), (r"\\bgj\\b", ("energy_kwh", 277.7778)),\n    (r"\\bmwp?\\b", ("power_mw", 1.0)), (r"\\bkwp?\\b", ("power_mw", 1e-3)),\n    (r"m3|m³|khối", ("volume_m3", 1.0)), (r"lít|\\bl\\b", ("volume_m3", 1e-3)),\n    (r"tấn|\\bt\\b|tons?|tonnes?", ("mass_t", 1.0)), (r"\\bkg\\b", ("mass_t", 1e-3)),\n    (r"%|phần trăm", ("percent", 1.0)), (r"\\bha\\b|hecta|héc", ("area_ha", 1.0)), (r"cây|trees?", ("count_tree", 1.0)),\n]\n\n\ndef unit_dimension(unit_text: str) -> tuple[str, float]:\n    """Dimension and factor of a unit; an intensity keeps its denominator (\'ghg_t/tấn thép\')."""\n    text = (unit_text or "").strip().lower()\n    num, _, den = text.partition("/")\n    dim, factor = next(((d, f) for pattern, (d, f) in _UNITS if re.search(pattern, num)), (num or "none", 1.0))\n    if den:\n        dim += "/" + re.sub(r"\\s+", " ", den.strip())\n    return dim, factor\n\n\ndef parse_number(text: str) -> tuple[float, float]:\n    """Value of a written number and half its last written digit (the published precision).\n\n    Vietnamese writes 22.540.603 and 23,5; some reports use 1,234.5. A separator followed by exactly\n    three digits is read as a thousands separator, any other as the decimal mark.\n    """\n    s = re.sub(r"\\s", "", text or "")\n    if not re.fullmatch(r"[\\d.,]+", s) or not re.search(r"\\d", s):\n        raise ValueError(f"not a number: {text!r}")\n    if "." in s and "," in s:\n        dec = "," if s.rfind(",") > s.rfind(".") else "."\n        integer, decimals = s.replace("." if dec == "," else ",", "").split(dec)\n    elif s.count(".") > 1 or s.count(",") > 1:\n        integer, decimals = re.sub(r"[.,]", "", s), ""\n    elif "." in s or "," in s:\n        integer, decimals = re.split(r"[.,]", s)\n        if len(decimals) == 3:\n            integer, decimals = integer + decimals, ""\n    else:\n        integer, decimals = s, ""\n    value = float(f"{integer or 0}.{decimals or 0}")\n    return value, 0.5 * 10 ** (-len(decimals))\n\n\n@dataclass\nclass IR:\n    metric: str\n    value_text: str\n    unit_text: str = ""\n    multiplier_text: str = ""\n    value_kind: str = "absolute"\n    change_direction: str | None = None  # decrease | increase, for value_kind == change\n    baseline_year: int | None = None\n    period: int | None = None\n    status: str = "actual"\n    scope: str | None = None\n    boundary: str = "unspecified"\n    metric_text: str = ""\n    subject_text: str = ""\n    extra: dict = field(default_factory=dict)\n\n    @classmethod\n    def from_dict(cls, d: dict) -> IR:\n        known = {k: d.get(k) for k in cls.__dataclass_fields__ if k in d and k != "extra"}\n        ir = cls(**known)\n        ir.boundary = ir.boundary or "unspecified"\n        ir.status = ir.status or "actual"\n        ir.value_kind = ir.value_kind or "absolute"\n        for name in ("period", "baseline_year"):\n            v = getattr(ir, name)\n            setattr(ir, name, int(v) if v not in (None, "", "null") and str(v).isdigit() else None)\n        return ir\n\n    def as_dict(self) -> dict:\n        return asdict(self)\n\n    def quantity(self) -> tuple[float, float, str]:\n        """Value and tolerance in the dimension\'s base unit, and the dimension."""\n        value, half = parse_number(self.value_text)\n        mult = MULTIPLIERS.get((self.multiplier_text or "").strip().lower(), 1.0)\n        dim, factor = unit_dimension(self.unit_text)\n        return value * mult * factor, half * mult * factor, dim\n\n\ndef comparable(claim: IR, evidence: IR) -> tuple[bool, str]:\n    """Same metric, value kind, status, period, scope, boundary and unit dimension (invariant 4)."""\n    checks = [\n        ("chỉ số", claim.metric, evidence.metric),\n        ("loại giá trị", claim.value_kind, evidence.value_kind),\n        ("thực hiện / mục tiêu", claim.status, evidence.status),\n        ("kỳ", claim.period, evidence.period),\n        ("phạm vi phát thải", claim.scope, evidence.scope),\n    ]\n    if claim.value_kind == "change":\n        checks.append(("năm gốc", claim.baseline_year, evidence.baseline_year))\n    if "unspecified" not in (claim.boundary, evidence.boundary):\n        checks.append(("ranh giới tổ chức", claim.boundary, evidence.boundary))\n    for name, a, b in checks:\n        if a != b:\n            return False, f"khác {name}: {a} / {b}"\n    try:\n        dim_c, dim_e = claim.quantity()[2], evidence.quantity()[2]\n    except ValueError as exc:\n        return False, f"không đọc được số: {exc}"\n    if dim_c != dim_e:\n        return False, f"khác đơn vị: {dim_c} / {dim_e}"\n    return True, "so được"\n\n\ndef compare(claim: IR, evidence: IR) -> tuple[str, str]:\n    """S (khớp), C (mâu thuẫn) or N (không so được), with the arithmetic or the reason."""\n    ok, why = comparable(claim, evidence)\n    if not ok:\n        return "N", why\n    if claim.value_kind == "change" and claim.change_direction != evidence.change_direction:\n        return "C", f"ngược chiều: {claim.change_direction} / {evidence.change_direction}"\n    vc, tc, dim = claim.quantity()\n    ve, te, _ = evidence.quantity()\n    gap = abs(vc - ve)\n    label = "S" if gap <= tc + te else "C"\n    return label, f"|{vc:g} − {ve:g}| = {gap:g} {dim}; dung sai {tc + te:g}"\n\n\ndef decide(claim: IR, evidence: list[IR]) -> tuple[str, list[tuple[str, str]]]:\n    """Verdict over every evidence figure: S if some comparable figure matches and none conflicts,\n    C if comparable figures only conflict, N otherwise (nothing comparable, or mixed: a person reads it)."""\n    steps = [compare(claim, e) for e in evidence]\n    labels = {label for label, _ in steps if label != "N"}\n    return (labels.pop() if len(labels) == 1 else "N"), steps\n',
    'ag1_worker.py': '"""AG-1 worker: one local LLM reads every claim and evidence text of GS-Bench v1 into IR figures; irlib decides.\n\n    python -u ag1_worker.py --job \'{"tag": "q3-4b-inst", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1, "max_len": 4096}\'\n\nRuns inside a Kaggle notebook built by build_nb.py (files embedded next to it: llm.py from ZAIC Experiment/common,\nirlib.py, gs_bench_v1.jsonl). Greedy, thinking off. The model only fills the IR; S / C / N comes from irlib.decide\n(arithmetic by code, invariant 5). Output: out/<tag>_texts.jsonl (raw + parsed figures per unique text) and\nout/<tag>_cases.jsonl (verdict per case), plus a one-line summary in the log.\nEnv: LIMIT=n (first n cases, smoke), MAX_TOKENS (default 768).\n"""\nimport argparse, hashlib, json, os, re, time\nfrom pathlib import Path\n\nfrom irlib import IR, METRICS, SCOPES, VALUE_KINDS, decide\n\nPROMPT = """Bạn trích số liệu môi trường từ MỘT câu trong báo cáo của doanh nghiệp Việt Nam.\nVới MỖI con số mô tả một chỉ số môi trường (không lấy số năm, số điều luật, số thứ tự, số tiền), trả về một đối tượng:\n- "metric": một mã trong {metrics}\n- "metric_text": cụm từ gọi tên chỉ số, chép NGUYÊN VĂN từ câu\n- "value_text": con số, chép NGUYÊN VĂN từ câu (giữ dấu chấm, phẩy), không kèm chữ\n- "multiplier_text": "nghìn", "triệu", "tỷ" nếu câu viết chữ đó ngay sau số, ngược lại ""\n- "unit_text": đơn vị nguyên văn (vd "tCO2e", "kWh", "m3", "tấn", "%", "tCO2e/tấn sản phẩm"); "" nếu không có\n- "value_kind": một trong {kinds} (share = tỷ lệ phần trăm của một tổng; change = mức tăng/giảm so với kỳ trước hoặc năm gốc; intensity = trên một đơn vị sản phẩm/doanh thu)\n- "change_direction": "decrease" | "increase" | null (chỉ khi value_kind = change)\n- "baseline_year": năm gốc nếu là change và câu có nêu, ngược lại null\n- "period": năm của số liệu nếu câu NÊU RÕ; null nếu câu không nêu\n- "status": "actual" (đã thực hiện) | "target" (mục tiêu, kế hoạch, cam kết)\n- "scope": một trong {scopes} nếu là phát thải khí nhà kính và câu nêu phạm vi, ngược lại null\n- "boundary": "group" (tập đoàn, hợp nhất) | "parent" (công ty mẹ) | "facility" (nhà máy, cơ sở cụ thể) | "unspecified"\nCâu không có số liệu môi trường thì trả về danh sách rỗng.\n\nCÂU: {sentence}\n\nChỉ trả lời đúng một đối tượng JSON: {{"figures": [ ... ]}}"""\n\n\ndef parse(text):\n    """First JSON object in the reply; None when there is none."""\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth = 0\n        for i in range(start, len(text)):\n            depth += {"{": 1, "}": -1}.get(text[i], 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef to_irs(figures):\n    out = []\n    for f in figures or []:\n        if isinstance(f, dict) and f.get("value_text") not in (None, ""):\n            try:\n                ir = IR.from_dict({**f, "value_text": str(f["value_text"])})\n                ir.quantity()  # unreadable number -> dropped\n                out.append(ir)\n            except (TypeError, ValueError):\n                pass\n    return out\n\n\ndef verdict(claim_irs, ev_irs):\n    if not claim_irs:\n        return "N", ["không đọc được số liệu trong tuyên bố"]\n    labels, why = [], []\n    for c in claim_irs:\n        label, steps = decide(c, ev_irs)\n        labels.append(label)\n        why.append(f"{c.value_text} {c.unit_text}: {label} " + "; ".join(s[1] for s in steps)[:300])\n    if "C" in labels:\n        return "C", why\n    return ("S" if all(x == "S" for x in labels) else "N"), why\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    cases = [json.loads(line) for line in open("gs_bench_v1.jsonl", encoding="utf-8")]\n    if os.environ.get("LIMIT"):\n        cases = cases[: int(os.environ["LIMIT"])]\n    texts = {}\n    for c in cases:\n        for t in (c["claim"], c["evidence"]):\n            texts.setdefault(hashlib.sha1(t.encode()).hexdigest()[:16], t)\n    print(f"[{tag}] {len(cases)} cases, {len(texts)} unique texts, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    keys = list(texts)\n    prompts = [[{"role": "user", "content": PROMPT.format(metrics=list(METRICS), kinds=list(VALUE_KINDS), scopes=list(SCOPES),\n                                                         sentence=texts[k])}] for k in keys]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "768")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)}", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    irs, bad = {}, 0\n    with open(f"out/{tag}_texts.jsonl", "w", encoding="utf-8") as fh:\n        for k, r in zip(keys, raw):\n            obj = parse(r)\n            bad += obj is None\n            figs = (obj or {}).get("figures") if isinstance((obj or {}).get("figures"), list) else []\n            irs[k] = to_irs(figs)\n            fh.write(json.dumps({"key": k, "text": texts[k], "raw": r, "figures": [x.as_dict() for x in irs[k]]}, ensure_ascii=False) + "\\n")\n    right = 0\n    with open(f"out/{tag}_cases.jsonl", "w", encoding="utf-8") as fh:\n        for c in cases:\n            kc, ke = (hashlib.sha1(t.encode()).hexdigest()[:16] for t in (c["claim"], c["evidence"]))\n            label, why = verdict(irs[kc], irs[ke])\n            right += label in c["labels"]\n            fh.write(json.dumps({"id": c["id"], "label": label, "why": why}, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "cases": len(cases), "right": right, "unparsed_texts": bad,\n               "load_s": round(load_s), "gen_s": round(gen_s), "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'gs_bench_v1.jsonl': '{"id": "ba-F01", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khớp sau làm tròn", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty là khoảng 23,5 triệu tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["S"]}\n{"id": "ba-F02", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khớp sau làm tròn", "claim": "Phát thải khí nhà kính phạm vi 2 của Công ty năm 2025 là 933.876 tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["S"]}\n{"id": "ba-F03", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty là 21,4 triệu tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["C"]}\n{"id": "ba-F04", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Phát thải khí nhà kính phạm vi 2 của Công ty năm 2025 là 833.876 tCO2e.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["C"]}\n{"id": "ba-F05", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "lệch một con số", "claim": "Năm 2024, nhà máy điện nhiệt dư đáp ứng 90% nhu cầu điện của khu liên hợp.", "evidence": "Năm 2024, nhà máy điện nhiệt dư đáp ứng 80% nhu cầu điện của khu liên hợp.", "labels": ["C"]}\n{"id": "ba-F06", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 800.000 tCO2e.", "labels": ["S"]}\n{"id": "ba-F07", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 850.000 tCO2e.", "labels": ["C"]}\n{"id": "ba-F08", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "tính lại % thay đổi", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 20% so với năm 2020.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty: năm 2020 là 1.000.000 tCO2e; năm 2025 là 1.050.000 tCO2e.", "labels": ["C"]}\n{"id": "ba-F09", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Hoạt động sản xuất gang thép chiếm hơn 99% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "Năm 2025, sản lượng thép thô của Tập đoàn tăng 24% so với năm 2024.", "labels": ["N"]}\n{"id": "ba-F10", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Cường độ phát thải khí nhà kính năm 2025 của Công ty là 1,9 tCO2e/tấn thép.", "evidence": "Bảng 5.2 — Phát thải khí nhà kính năm 2025: Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e; Tổng phạm vi 1 và 2: 23.474.480 tCO2e.", "labels": ["N"]}\n{"id": "ba-F11", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác cơ sở đo", "claim": "Tổng phát thải khí nhà kính của Công ty năm 2025 giảm 5% so với năm 2024.", "evidence": "Cường độ phát thải khí nhà kính (tCO2e/tấn thép) của Công ty năm 2025 giảm 5% so với năm 2024.", "labels": ["N"]}\n{"id": "ba-F12", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác phạm vi phát thải", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty giảm 30% so với năm 2020.", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1, 2 và 3 của Công ty giảm 8% so với năm 2020.", "labels": ["N"]}\n{"id": "ba-F13", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác phạm vi phát thải", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 23,5 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 của Công ty năm 2025 là 22.540.603 tCO2e.", "labels": ["N"]}\n{"id": "ba-F14", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác kỳ", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 của Công ty là 22,5 triệu tCO2e.", "evidence": "Năm 2024, phát thải khí nhà kính phạm vi 1 của Công ty là 22,5 triệu tCO2e.", "labels": ["N"]}\n{"id": "ba-F15", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác kỳ", "claim": "Năm 2025, lượng nước sử dụng của Công ty giảm 12% so với năm 2024.", "evidence": "Năm 2023, lượng nước sử dụng của Công ty giảm 12% so với năm 2022.", "labels": ["N"]}\n{"id": "ba-F16", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Công ty đã giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "evidence": "Mục tiêu đến năm 2030: giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "labels": ["N"]}\n{"id": "ba-F17", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Công ty đặt mục tiêu giảm 30% phát thải khí nhà kính phạm vi 1 và 2 vào năm 2030 so với năm 2020.", "evidence": "Mục tiêu đến năm 2030: giảm 30% phát thải khí nhà kính phạm vi 1 và 2 so với năm 2020.", "labels": ["S"]}\n{"id": "ba-F18", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "mục tiêu hay thực hiện", "claim": "Năm 2025, tỷ lệ phế liệu tái chế đưa vào sản xuất của Công ty đạt 30%.", "evidence": "Năm 2025, tỷ lệ phế liệu tái chế đưa vào sản xuất của Công ty đạt 18%, thấp hơn mục tiêu 30%.", "labels": ["C"]}\n{"id": "ba-F19", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác ranh giới tổ chức", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 hợp nhất của Tập đoàn năm 2025 là 23,5 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 và 2 của Khu liên hợp Dung Quất năm 2025 là 12.300.000 tCO2e.", "labels": ["N"]}\n{"id": "ba-F20", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác ranh giới tổ chức", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 của công ty mẹ năm 2025 là 1,2 triệu tCO2e.", "evidence": "Phát thải khí nhà kính phạm vi 1 và 2 hợp nhất toàn Tập đoàn năm 2025 là 23,47 triệu tCO2e.", "labels": ["N"]}\n{"id": "ba-F21", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty tiêu thụ 5,6 triệu MWh điện.", "evidence": "Năm 2025, Công ty tiêu thụ 5,6 triệu tấn than.", "labels": ["N"]}\n{"id": "ba-F22", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty sử dụng 15 triệu m3 nước.", "evidence": "Lượng nước sử dụng của Công ty năm 2025: 15.000 m3.", "labels": ["C"]}\n{"id": "ba-F23", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "khác đơn vị / hệ số", "claim": "Năm 2025, Công ty sử dụng 15 triệu m3 nước.", "evidence": "Lượng nước sử dụng của Công ty năm 2025: 15.000.000 m3.", "labels": ["S"]}\n{"id": "ba-F24", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "dung sai theo độ chính xác", "claim": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025 đạt 12%.", "evidence": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025: 12,4%.", "labels": ["S"]}\n{"id": "ba-F25", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "dung sai theo độ chính xác", "claim": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025 đạt 12,0%.", "evidence": "Tỷ lệ năng lượng tái tạo trong tổng năng lượng tiêu thụ của Công ty năm 2025: 12,4%.", "labels": ["C"]}\n{"id": "ba-F26", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cộng lại bảng", "claim": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 23.474.480 tCO2e.", "evidence": "Phát thải khí nhà kính năm 2025 của Công ty — Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e.", "labels": ["S"]}\n{"id": "ba-F27", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cộng lại bảng", "claim": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty năm 2025 là 24.474.480 tCO2e.", "evidence": "Phát thải khí nhà kính năm 2025 của Công ty — Phạm vi 1: 22.540.603 tCO2e; Phạm vi 2: 933.876 tCO2e.", "labels": ["C"]}\n{"id": "ba-F28", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "bẫy từ phủ định", "claim": "Năm 2024, Tập đoàn giảm phát thải khí nhà kính tại các nhà máy thép.", "evidence": "Năm 2024, nhà máy thép áp dụng công nghệ thiêu kết mới giúp giảm phát thải khí nhà kính.", "labels": ["N", "S"]}\n{"id": "ba-F29", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "bẫy từ phủ định", "claim": "Năm 2024, Tập đoàn giảm phát thải khí nhà kính tại các nhà máy thép.", "evidence": "Năm 2024 không có sự cố môi trường nào tại các nhà máy thép; phát thải khí nhà kính giảm so với năm trước.", "labels": ["N", "S"]}\n{"id": "ba-F30", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "không có bằng chứng liên quan", "claim": "Năm 2025, 100% nước thải của Công ty được xử lý đạt quy chuẩn trước khi xả ra môi trường.", "evidence": "Năm 2025, Công ty tổ chức 12 khoá đào tạo an toàn lao động cho 3.000 lượt người lao động.", "labels": ["N"]}\n{"id": "ba-F31", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "không có bằng chứng liên quan", "claim": "Công ty đã đạt trung hòa carbon trong năm 2025.", "evidence": "Năm 2025, Công ty trồng 50.000 cây xanh tại các khu công nghiệp.", "labels": ["N"]}\n{"id": "ba-F32", "layer": "B-a sạch", "company": "dựng", "sector": "thép", "family": "cùng số, khác chỉ số", "claim": "Năm 2025, Công ty tự chủ 90% nhu cầu điện cho sản xuất thép.", "evidence": "Năm 2025, Công ty hoàn thành 90% kế hoạch kiểm kê khí nhà kính.", "labels": ["N"]}\n{"id": "ba-N01", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 toàn Tập đoàn là 23,47 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N02", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 toàn Tập đoàn là 23.369.500 tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["C"]}\n{"id": "ba-N03", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng thép phát thải 23.369.500 tCO2e, chiếm 99,55% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N04", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng nông nghiệp chiếm 0,39% tổng phát thải khí nhà kính của Tập đoàn.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N05", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, mảng nông nghiệp phát thải 933.876 tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["C"]}\n{"id": "ba-N06", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 3 toàn Tập đoàn là 23,47 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["N"]}\n{"id": "ba-N08", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "bảng dàn phẳng", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 2 toàn Tập đoàn là 0,93 triệu tCO2e.", "evidence": "39\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\n THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\nTRONG LĨNH VỰC NÔNG NGHIỆP\\nTrong các nhóm ngành kinh doanh của Tập đoàn Hòa Phát, biến đổi khí hậu tác \\nđộng rõ nét nhất đến lĩnh vực nông nghiệp — đặc biệt là hoạt động chăn nuôi. \\nNhiệt độ tăng cao và các đợt thời tiết cực đoan khiến gia súc, gia cầm chịu stress \\nnhiệt, dẫn tới giảm tốc độ tăng trưởng, giảm khả năng sinh sản và năng suất. Thêm \\nvào đó, các hiện tượng như thiên tai, bão lũ và dịch bệnh tăng tần suất càng làm \\ntăng rủi ro cho chuỗi sản xuất chăn nuôi. Theo báo cáo của World Bank Group \\n– Asian Development Bank. nông-nghiệp Việt Nam có thể chịu tổn thất khoảng \\n5.6-6.2% GDP vào năm 2030, và 7.6-10.6% GDP vào năm 2050 nếu không có các \\nbiện pháp ứng phó hiệu quả. \\nĐể chủ động ứng phó với biến đổi khí hậu, các công ty thuộc nhóm Nông nghiệp \\ncủa Hòa Phát đã triển khai hàng loạt giải pháp công nghệ tiên tiến nhằm thích \\nứng với điều kiện khí hậu thay đổi. Hệ thống chuồng kín điều khiển tự động được \\náp dụng rộng rãi tại các trang trại do công ty sở hữu và quản lý, cho phép kiểm \\nsoát chính xác nhiệt độ, độ ẩm và tốc độ gió, đảm bảo môi trường sống ổn định, \\nhạn chế tác động của biến động thời tiết lên sức khỏe và tâm lý vật nuôi. Phòng \\nấp trứng làm mát chuyên dụng được vận hành, giúp duy trì nhiệt độ và độ ẩm \\ntối ưu cho sự phát triển của phôi, tăng tỷ lệ nở và chất lượng con giống. Thiết \\nlập khẩu phần dinh dưỡng tối ưu cho từng giống vật nuôi, từng giai đoạn tăng \\ntrưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của \\nbiến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải \\nqua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) \\ntrong chăn nuôi, thu hồi khí gas để vận hành máy phát điện và đun nấu... Những \\ngiải pháp này không chỉ giúp giảm thiểu tác động tiêu cực của biến đổi khí hậu \\nmà còn nâng cao phúc lợi động vật, cải thiện hiệu quả chăn nuôi và đảm bảo an \\ntoàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp \\nbền vững và thích ứng khí hậu. \\nKIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 \\n1. Tổng phát thải Khí nhà kính toàn Tập đoàn kiểm kê theo ISO 14064 cho năm 2025:\\n2. Phát thải theo tổng công ty năm 2025\\nPhạm vi 1\\nPhạm vi 2\\nCộng\\nThép\\nNông nghiệp\\nBất động sản\\nĐiện máy gia dụng\\nTòa nhà văn phòng\\nCộng\\n22.540.603\\n933.876\\n23.474.480\\n23.369.500 \\n90.846 \\n2.836 \\n9.666 \\n1.631 \\n23.474.480 \\n99,55% \\n0,39% \\n0,01% \\n0,04% \\n0,01% \\n100% \\nPHẠM VI\\nNHÓM NGÀNH\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nPHÁT THẢI TOÀN TẬP ĐOÀN\\nNĂM 2025 (tCO2e)\\nTỶ TRỌNG\\n KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 \\nHÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU\\n5.1\\nHành động giảm thiểu và\\nthích ứng với biến đổi khí hậu\\nQuản lý môi trường\\nPHẦN 5\\nXANH HÓA\\nSẢN XUẤT\\n", "labels": ["S"]}\n{"id": "ba-N09", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, Tập đoàn hoàn thành 90% kế hoạch kiểm kê khí nhà kính.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C", "N"]}\n{"id": "ba-N10", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Khu liên hợp Gang thép Hòa Phát Dung Quất tự chủ 90% nhu cầu điện.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["S"]}\n{"id": "ba-N11", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Khu liên hợp Gang thép Hòa Phát Dung Quất tái sử dụng 90% nước.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C"]}\n{"id": "ba-N12", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, tỷ lệ tái chế, tái sử dụng chất thải rắn của nhóm thép và nông nghiệp đạt trên 99%.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["S"]}\n{"id": "ba-N13", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "chữ dính cột", "claim": "Năm 2025, tỷ lệ phụ nữ ở các cấp quản lý trong toàn Tập đoàn đạt 22%.", "evidence": "9\\nBÁO CÁO PHÁT TRIỂN BỀN VỮNG \\n2025\\nĐóng góp gần 13.000 tỷ đồng vào \\nngân sách Nhà nước  \\nĐóng góp hơn 120 tỷ đồng cho các \\nchương trình phúc lợi xã hội.\\nMÔI TRƯỜNG\\nXÃ HỘI\\nQUẢN TRỊ\\nHoàn thành kiểm kê khí nhà kính theo tiêu \\nchuẩn ISO 14064-1:2018 cho toàn Tập \\nđoàn và được BSI thẩm định số liệu.  \\nTái chế, tái sử dụng chất thải rắn\\nnhóm thép và nông nghiệp đạt trên 99% \\nSố giờ đào tạo trung bình cho mỗi nhân \\nviên là 38,52 giờ mỗi năm \\n22% thành viên\\nHội đồng Quản trị là thành viên độc lập.  \\nĐiểm nổi bật về\\nphát triển bền vững năm 2025\\nGiải thưởng\\nvà vinh danh năm 2025\\nThành viên của\\ncác Hiệp hội ngành nghề\\nHoạt động kinh doanh\\ntại Tập đoàn\\nPHẦN 3\\nTỔNG QUAN\\nVỀ TẬP ĐOÀN \\nHÒA PHÁT\\nĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025\\n3.1\\nTự chủ 90% nhu cầu điện\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTái sử dụng 98% nước\\ntại Khu liên hợp Gang thép Hòa Phát \\nDung Quất\\nTỷ lệ phụ nữ ở các cấp quản lý trong toàn \\nTập đoàn đạt 35,85% vào năm 2025\\n", "labels": ["C"]}\n{"id": "ba-N14", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, các nhà máy điện nhiệt dư tại hai khu liên hợp Hải Dương và Dung Quất phát 3,18 tỷ kWh điện, tăng 29% so với năm trước.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "ba-N15", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, nhà máy điện nhiệt dư tại Dung Quất phát 3,18 tỷ kWh điện.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C"]}\n{"id": "ba-N16", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, lượng điện phát của các nhà máy điện nhiệt dư thuộc hai khu liên hợp tăng 39% so với cùng kỳ.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C"]}\n{"id": "ba-N17", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, Hòa Phát tự chủ 100% lượng điện cho sản xuất thép.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["C", "N"]}\n{"id": "ba-N18", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Công nghệ tuabin thu hồi năng lượng quạt gió lò cao (BPRT) giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương pháp truyền thống.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "ba-N19", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Năm 2024, giải pháp thu hồi nhiệt dư giúp Hòa Phát giảm 50% phát thải CO2.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["N"]}\n{"id": "ba-N20", "layer": "B-a trang thật", "company": "hpg", "sector": "thép", "family": "nhiều số, nhiều ranh giới", "claim": "Ngày 05/02/2025, nhà máy nhiệt điện tại Dung Quất đạt sản lượng điện lũy kế 10 tỷ kWh.", "evidence": "86\\n  TỰ CHỦ 90% ĐIỆN SẢN XUẤT THÉP \\nNăm 2024, các nhà máy điện nhiệt dư thuộc hai Khu liên hợp sản xuất gang \\nthép Hòa Phát tại Hải Dương và Dung Quất có tổng lượng điện phát đạt 3,18 \\ntỷ kWh, tăng 29% so với cùng kỳ, giúp Hòa Phát tự chủ trên 90% lượng điện \\ncho sản xuất. Riêng tại Dung Quất, ngày 05/02/2025, Nhà máy Nhiệt điện – \\nCông ty Thép Hòa Phát Dung Quất chính thức cán mốc sản lượng điện lũy kế \\ntừ khi hoạt động đến nay là 10 tỷ kWh, góp phần tối ưu hóa chi phí sản xuất.\\n   TỰ CHỦ HƠN 90% ĐIỆN CHO SẢN XUẤT, GIẢM PHÁT \\n  THẢI KHÍ NHÀ KÍNH, MANG LẠI LỢI ÍCH KINH TẾ LỚN\\nTrong năm qua, sản lượng điện tự phát của Hòa Phát Dung Quất đạt 2,39 tỷ kWh, \\ntự chủ hơn 90% nhu cầu sản xuất gang thép toàn Khu liên hợp. Sản lượng phát điện \\ntự dùng cung cấp cho Khu liên hợp Thép Hòa Phát Hải Dương đạt 788 triệu kWh. \\nTổng lượng điện phát tự chủ được của Hòa Phát tại hai Khu liên hợp Hải Dương và \\nDung Quất tương đương một nhà máy nhiệt điện tầm trung, quy đổi theo giá điện \\nhiện hành, sản lượng phát điện năm 2024 có giá trị khoảng 5.400 tỷ đồng. \\nViệc thu hồi và sử dụng nhiệt dư, khí dư để phát điện là giải pháp tiết kiệm năng \\nlượng hiệu quả, tăng tính cạnh tranh cho giá thành sản phẩm, giảm thiểu ô nhiễm \\nmôi trường, đồng thời giảm tải áp lực điều độ hệ thống điện lưới quốc gia.\\nCác Khu liên hợp gang thép của Hòa Phát được đầu tư dây chuyền thiết bị hiện đại \\ncủa các nước G7 giúp tối ưu hóa sản xuất, tận dụng mọi nguồn nhiệt, khí dư thừa \\ntrong quá trình luyện gang thép để nâng cao hiệu suất phát điện, tiết kiệm chi phí \\nsản xuất, phát triển bền vững.\\nHiện nay, Hòa Phát đang áp dụng 5 giải pháp sản xuất thép tiết kiệm năng lượng.\\nGiải pháp thứ nhất là thu hồi nhiệt dư, khí nóng lò cốc, lò cao, lò thổi sản xuất \\nđiện. Nhiệt dư phát sinh từ quá trình sản xuất than cốc, khí dư từ quá trình luyện \\ngang, luyện thép được thu hồi, tận dụng phục vụ cho phát điện để cung cấp cho \\nsản xuất. \\nHai là sử dụng công nghệ dập cốc khô thân thiện với môi trường để sản xuất điện. \\nBa là công nghệ tuabin thu hồi năng lượng Quạt gió lò cao (BPRT), vận hành các \\nquạt gió công suất lớn, giúp tiết kiệm đến 50% lượng điện tiêu thụ so với phương \\npháp truyền thống.\\nBốn là sử dụng nhiệt dư để sản xuất điện trong quá trình thiêu kết quặng.\\nNăm là công nghệ Đúc – Cán liên tục, tận dụng nhiệt từ phôi nóng, phôi nóng sau \\nđúc có nhiệt độ 750-900 độ C sẽ được chuyển ngay sang dây chuyền cán để sản \\nxuất thép thành phẩm. Giải pháp này đã tiết giảm tối đa sử dụng năng lượng, làm \\ngiảm chi phí sản xuất, đồng thời giảm phát thải CO2/tấn sản phẩm.\\n    NHÀ MÁY NHIỆT ĐIỆN, THÉP HÒA PHÁT DUNG QUẤT\\n   CHÍNH THỨC CÁN MỐC 10 TỶ KWH\\nAnh Bùi Văn Hữu, Giám đốc Nhà máy Nhiệt điện, Công ty Thép Hòa Phát Dung \\nQuất cho biết: “Hạng mục phát điện nhiệt dư không chỉ giúp tiết kiệm năng lượng \\nmà còn góp phần quan trọng trong chiến lược giảm phát thải khí nhà kính, hướng \\nđến mục tiêu trung hòa carbon. Đây là giải pháp bền vững, tận dụng nguồn nhiệt \\ndư trong quá trình sản xuất gang thép để tạo ra điện mà không cần sử dụng nhiên \\nliệu hóa thạch”.\\nTừ tháng 7/2019, một năm sau khi lắp đặt tổ máy phát đầu tiên, sản lượng phát \\nđiện của Nhà máy đã đáp ứng nhu cầu sản xuất gang thép tổng khu liên hợp Hòa \\nPhát Dung Quất. Từ đó đến nay, sản lượng phát điện của Nhà máy tăng trưởng \\nhàng năm, đạt mốc 5 tỷ kWh vào ngày 19/9/2022 và cán mốc 10 tỷ kWh ngày \\n05/02/2025.\\nVới sự mở rộng của dự án Hòa Phát Dung Quất 2, công ty tiếp tục đầu tư mạnh \\nmẽ vào dây chuyền phát điện nhiệt dư thứ hai. Theo đó, Nhà máy Nhiệt điện số 2 \\nsở hữu công suất 240MW, gồm 04 tổ máy phát 60MW, nâng tổng công suất phát \\nđiện của toàn khu liên hợp lên 600MW.\\nBÁO CÁO THƯỜNG NIÊN - 2024\\nPHẦN 6\\nPHÁT TRIỂN\\nBỀN VỮNG\\nHòa Phát hoàn thành kiểm kê khí \\nnhà kính toàn tập đoàn\\nSáng kiến cải tiến kỹ thuật góp \\nphần tiết kiệm hàng trăm tỷ đồng\\nTự chủ 90% điện sản xuất thép\\nPhát triển sản phẩm mới\\nChuyển đổi số tại tập đoàn \\nHòa Phát năm 2024\\nTrách nhiệm xã hội - cộng đồng\\n", "labels": ["S"]}\n{"id": "bb-2a31912f6d377a83-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 34,3 nghìn m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "34,3", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 48.055 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "48.055", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 nghìn m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 34.325 m3 vào năm 2030.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 34.325 m3/tấn sản phẩm.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2a31912f6d377a83-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 34.325 m3.", "evidence": "SỬ DỤNG TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 34.325 m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "34.325", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Mức tiêu thụ nước của Công ty giảm 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Mức tiêu thụ nước của Công ty giảm 27,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "27,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Mức tiêu thụ nước của Công ty tăng 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Mức tiêu thụ nước giảm 12,8% vào năm 2030.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-f5f8a5c13872aea7-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty giảm 12,8% so với năm 2024.", "evidence": "Mức tiêu thụ nước giảm : 12,8% (so với năm 2024: 4.878 m3) VÀ PHÁT THẢI HIỆU QUẢ • Phun nước làm ẩm đường vào những ngày khô hanh.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12,8", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Mức tiêu thụ nước", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9c78bf02decaf05d-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 39% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 54% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "54", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty tăng 39% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ gas và xăng dầu giảm 39% vào năm 2030.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "39", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 14% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty giảm 29% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "29", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Năm 2021, mức tiêu thụ gas và xăng dầu của Công ty tăng 14% so với năm 2020.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-9c78bf02decaf05d-1-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ gas và xăng dầu giảm 14% vào năm 2030.", "evidence": "Trong năm 2021, mặc dù tổng sản lượng của PNJ tăng lên 20% để đáp ứng nhu cầu gia tăng của thị trường, mức tiêu thụ gas và xăng dầu giảm tương ứng là 39% và 14% so với năm 2020 NHU CẦU VỀ ĐIỆN NĂNG VÀ NHIÊN LIỆU MỨC TIÊU THỤ NĂNG LƯỢNG Mức sử dụng điện năng lượng, nhiên liệu qua các năm", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "14", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2020, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ gas và xăng dầu", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-270002ea10dc9ad7-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 10,7 nghìn m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "10,7", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 8.584 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "8.584", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 nghìn m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 10.730 m3 vào năm 2030.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 10.730 m3/tấn sản phẩm.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-270002ea10dc9ad7-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 10.730 m3.", "evidence": "TÀI NGUYÊN NƯỚC HIỆU QUẢ Lượng nước đầu vào theo nguồn Tổng lượng nước sử dụng: 10.730 m3 Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị cung cấp.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "10.730", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty giảm 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty giảm 21% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "21", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Năm 2014, tổng tiêu thụ điện của Công ty tăng 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu tổng tiêu thụ điện giảm 6% vào năm 2030.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-9796bc7d982bb3ce-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Năm 2014, lượng điện từ năng lượng tái tạo của Công ty giảm 6% so với năm 2013.", "evidence": "TIÊU THỤ TÀI NGUYÊN ĐIỆN, NƯỚC, GIẤY Năm 2014, tổng tiêu thụ điện khu vực văn phòng giảm 6% so với năm 2013, duy trì tiết kiệm điện nước, ổn định mức tiêu thụ điện nước cho sản xuất, lượng giấy sử dụng cho toàn bộ hoạt động của Công ty.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "6", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2013, "period": 2014, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng tiêu thụ điện", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-S1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S1 rounded", "claim": "Tổng lượng nước sử dụng của Công ty là 12,7 nghìn m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "12,7", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 15.908 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "15.908", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 nghìn m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 12.726 m3 vào năm 2030.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 12.726 m3/tấn sản phẩm.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c5b8455b08f5a561-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 12.726 m3.", "evidence": "Sử dụng tài nguyên nước hiệu quả • Tổng lượng nước sử dụng: 12.726 m3;", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "12.726", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d7a04b919bbe8cda-1-S0", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "S0 restate", "claim": "Lượng nước tiêu thụ của Công ty giảm 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-C1", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C1 value", "claim": "Lượng nước tiêu thụ của Công ty giảm 56% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "56", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-C3", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "C3 direction", "claim": "Lượng nước tiêu thụ của Công ty tăng 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-N2", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N2 status", "claim": "Công ty đặt mục tiêu lượng nước tiêu thụ giảm 41% vào năm 2030.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-d7a04b919bbe8cda-1-N6", "layer": "B-b", "company": "bvh", "sector": "bảo hiểm", "family": "N6 metric", "claim": "Lượng nước thải của Công ty giảm 41% so với năm 2024.", "evidence": "nước sinh hoạt chiếm tỷ trọng lớn, khoảng 4.849 m3, trong khi nước uống tinh khiết So với năm 2024, lượng nước tiêu thụ ghi nhận xu hướng giảm đáng kể khoảng 41%, cho thấy hiệu quả của các giải pháp kiểm soát và tối ưu sử dụng nước trong toàn hệ thống.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "41", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2024, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 4}\n{"id": "bb-db8841dc88c6b9be-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tổng lượng nước sử dụng của Công ty là 5.692 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "5.692", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-C2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C2 magnitude", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 nghìn m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước sử dụng đạt 8.131 m3 vào năm 2030.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N5", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N5 basis", "claim": "Tổng lượng nước sử dụng của Công ty là 8.131 m3/tấn sản phẩm.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-db8841dc88c6b9be-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 8.131 m3.", "evidence": "Lượng nước đầu vào theo nguồn • Tổng lượng nước sử dụng: 8.131 m3 • Các số liệu được cập nhật từ chỉ số đồng hồ đo trực tiếp lưu lượng nước của đơn vị • Đơn vị cấp nước: Công ty TNHH một thành viên nước sạch Hà Nội.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "8.131", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước sử dụng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 46,97 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "46,97", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 tỷ Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "tỷ", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2022, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Điện năng tiêu thụ đạt 33,55 triệu Kwh vào năm 2030.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2023, Điện năng tiêu thụ của Công ty là 33,55 triệu Kwh/tấn sản phẩm.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh/tấn sản phẩm", "multiplier_text": "triệu", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-7843554e8927d787-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2023, lượng điện từ năng lượng tái tạo của Công ty là 33,55 triệu Kwh.", "evidence": "Tiêu thụ năng lượng tại DHG Pharma bao gồm các chỉ tiêu sau: Điện năng tiêu thụ năm 2023: 33,55 triệu Kwh (tương ứng 59,24 tỷ đồng chi phí).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "33,55", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Điện năng tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2025, nước thải của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-S1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S1 rounded", "claim": "Năm 2025, nước thải của Công ty là 159,1 nghìn m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "159,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2025, nước thải của Công ty là 127.283 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "127.283", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2025, nước thải của Công ty là 159.104 nghìn m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2024, nước thải của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2024, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 159.104 m3 vào năm 2030.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2025, nước thải của Công ty là 159.104 m3/tấn sản phẩm.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ee4b9253a749cd8a-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2025, lượng nước sử dụng của Công ty là 159.104 m3.", "evidence": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "159.104", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2025, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Năng lượng xanh và sạch của Công ty đạt 87%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Năng lượng xanh và sạch của Công ty đạt 72%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "72", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu năng lượng xanh và sạch đạt 87% vào năm 2030.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-631428f0da19badc-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty đạt 87%.", "evidence": "Tỷ lệ sử dụng năng lượng xanh và sạch trong hoạt động sản xuất đạt 87%.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "87", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng xanh và sạch", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Mức tiêu thụ điện năng của Công ty giảm 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["S"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Mức tiêu thụ điện năng của Công ty giảm 25,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "25,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-C3", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C3 direction", "claim": "Mức tiêu thụ điện năng của Công ty tăng 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["C"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu mức tiêu thụ điện năng giảm 10,1% vào năm 2030.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-e14c97c34811ab03-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng điện từ năng lượng tái tạo của Công ty giảm 10,1% so với năm 2016.", "evidence": "So với năm 2016 mức tiêu thụ điện năng tiêu giảm được 10,1% (năm 2016: 2.686.080 kW), một phần trong vấn đề giảm tiêu hao năng lượng là do có sự chuyển dịch về sản xuất sản phẩm sang 2 nhà máy Traphaco Hưng Yên và Traphaco CNC và áp dụng các biện pháp giảm tiêu hao năng lượng nhằm phát triển môi trường bền vững.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng điện từ năng lượng tái tạo", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "electricity_consumption", "value_text": "10,1", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2016, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức tiêu thụ điện năng", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-S0", "layer": "B-b", "company": "tra", "sector": "dược", "family": "S0 restate", "claim": "Tỷ lệ rác thải được tái chế của Công ty đạt 75%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["S"], "claim_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-C1", "layer": "B-b", "company": "tra", "sector": "dược", "family": "C1 value", "claim": "Tỷ lệ rác thải được tái chế của Công ty đạt 90%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["C"], "claim_ir": {"metric": "waste_recycled", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-N2", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tỷ lệ rác thải được tái chế đạt 75% vào năm 2030.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["N"], "claim_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c385bf4d13ea40ed-0-N6", "layer": "B-b", "company": "tra", "sector": "dược", "family": "N6 metric", "claim": "Lượng chất thải phát sinh của Công ty đạt 75%.", "evidence": "»» Thực hiện vận chuyển định kỳ chất thải thông thường, chất thải tái chế theo quy Tỷ lệ rác thải được tái chế chiếm 75% khối lượng rác thải của Công ty.", "labels": ["N"], "claim_ir": {"metric": "waste_generated", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng chất thải phát sinh", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "waste_recycled", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ rác thải được tái chế", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 100% vào năm 2030.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-aa42b783af8c2d31-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% nước thải từ sản xuất và sinh hoạt được xử lý đạt chuẩn trước khi xả thải.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Hệ thống năng lượng mặt trời của Công ty là 2,2 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "2,2", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 nghìn MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu hệ thống năng lượng mặt trời đạt 1,8 MWp vào năm 2030.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Hệ thống năng lượng mặt trời của Công ty là 1,8 MWp/tấn sản phẩm.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-34d38386ad1c87b3-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty là 1,8 MWp.", "evidence": "Tính đến hết năm 2023, hệ thống năng lượng mặt trời với công suất 1,8 MWp đã được lắp đặt hoàn thiện ở các mái nhà xưởng, nhà ăn và kho thành phẩm, và bắt đầu vận hành Nhận thấy rằng nhiều khu vực trong nhà máy sản xuất khi không có người hay không sử dụng nếu vẫn dùng thiết bị điện cho tất cả các khu vực này sẽ gây lãng phí điện.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "1,8", "unit_text": "MWp", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "hệ thống năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-S1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S1 rounded", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125,1 nghìn m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "125,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 87.576 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "87.576", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 nghìn m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Lượng nước trong hoạt động sản xuất đạt 125.108 m3 vào năm 2030.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N5", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N5 basis", "claim": "Lượng nước trong hoạt động sản xuất của Công ty là 125.108 m3/tấn sản phẩm.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8e42f94aa937f2ed-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước thải của Công ty là 125.108 m3.", "evidence": "125.108 m3 Lượng nước trong hoạt động sản xuất.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "125.108", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Lượng nước trong hoạt động sản xuất", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu của Công ty đạt 86,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu của Công ty đạt 71,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "71,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tỷ lệ năng lượng xanh, sạch/nhiên liệu đạt 86,8% vào năm 2030.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-ebf98e6bd3bc6405-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty đạt 86,8%.", "evidence": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu: 86,8% Trụ sở chính, 11 Nhà Máy, 13 Trang trại được lắp đặt Hệ thống năng lượng mặt trời.", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "86,8", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tỷ lệ năng lượng xanh, sạch/nhiên liệu", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-c46b97cef8117154-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, tỷ lệ tái sử dụng nước của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, tỷ lệ tái sử dụng nước của Công ty đạt 75%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "75", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N1 period", "claim": "Năm 2020, tỷ lệ tái sử dụng nước của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu tỷ lệ tái sử dụng nước đạt 90% vào năm 2030.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-0-N6", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N6 metric", "claim": "Năm 2021, lượng nước thải của Công ty đạt 90%.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "90", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tỷ lệ tái sử dụng nước", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 3.200 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "3.200", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-C2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C2 magnitude", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 nghìn m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N1 period", "claim": "Năm 2020, lượng nước tái sử dụng của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2020, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu lượng nước tái sử dụng đạt 4000 m3 vào năm 2030.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N5", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N5 basis", "claim": "Năm 2021, lượng nước tái sử dụng của Công ty là 4000 m3/tấn sản phẩm.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-c46b97cef8117154-1-N6", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N6 metric", "claim": "Năm 2021, lượng nước thải của Công ty là 4000 m3.", "evidence": "Với sáng kiến trên, trong năm 2021, tỷ lệ tái sử dụng nước đạt khoảng 90% với lượng nước tái sử dụng đạt 4000m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "4000", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2021, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước tái sử dụng", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-11bff39daa1d0a60-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Nước thải đạt 100% vào năm 2030.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-11bff39daa1d0a60-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% Nước thải trong hoạt động Sản xuất và Chăn nuôi được xử lý đạt chuẩn.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-S1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S1 rounded", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253,1 nghìn m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["S"], "claim_ir": {"metric": "water_use", "value_text": "253,1", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 354.403 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "354.403", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-C2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C2 magnitude", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 nghìn m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["C"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N1 period", "claim": "Năm 2022, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2022, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu Tổng lượng nước tiêu thụ đạt 253.145 m3 vào năm 2030.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N5", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N5 basis", "claim": "Năm 2023, Tổng lượng nước tiêu thụ của Công ty là 253.145 m3/tấn sản phẩm.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-2d1b0b7ae3d3450d-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Năm 2023, lượng nước thải của Công ty là 253.145 m3.", "evidence": "Tổng lượng nước tiêu thụ năm 2023: 253.145 m3.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "water_use", "value_text": "253.145", "unit_text": "m3", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2023, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Tổng lượng nước tiêu thụ", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-S0", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-C1", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-N2", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N2 status", "claim": "Công ty đặt mục tiêu nước thải đạt 100% vào năm 2030.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-5d74fc3fa628f4a0-0-N6", "layer": "B-b", "company": "dhg", "sector": "dược", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-8a12299d98291537-0-S0", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "S0 restate", "claim": "Tiêu thụ gas của Công ty giảm 1,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["S"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-C1", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C1 value", "claim": "Tiêu thụ gas của Công ty giảm 16,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "16,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-C3", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "C3 direction", "claim": "Tiêu thụ gas của Công ty tăng 1,4% so với năm 2021.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["C"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "increase", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-8a12299d98291537-0-N2", "layer": "B-b", "company": "pnj", "sector": "bán lẻ trang sức", "family": "N2 status", "claim": "Công ty đặt mục tiêu tiêu thụ gas giảm 1,4% vào năm 2030.", "evidence": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "labels": ["N"], "claim_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "tiêu thụ gas", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "fuel_consumption", "value_text": "1,4", "unit_text": "%", "multiplier_text": "", "value_kind": "change", "change_direction": "decrease", "baseline_year": 2021, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "mức giảm tiêu thụ gas", "subject_text": "", "extra": {}}, "fold": 1}\n{"id": "bb-d08482796aa8de43-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Nước thải của Công ty đạt 100%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["S"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Nước thải của Công ty đạt 85%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["C"], "claim_ir": {"metric": "wastewater", "value_text": "85", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu Nước thải đạt 100% vào năm 2030.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-d08482796aa8de43-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Lượng nước sử dụng của Công ty đạt 100%.", "evidence": "Không có khiếu nại của cộng đồng xung quanh về Không có khiếu nại về bất bình đẳng Không có khiếu nại về phân biệt đối xử Không lao động cưỡng bức, không lao động trẻ em 100% Nước thải hoạt động sản xuất được xử lý trước khi thải ra môi trường.", "labels": ["N"], "claim_ir": {"metric": "water_use", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "lượng nước sử dụng", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "wastewater", "value_text": "100", "unit_text": "%", "multiplier_text": "", "value_kind": "share", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "Nước thải", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-S0", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S0 restate", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-S1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "S1 rounded", "claim": "Năng lượng mặt trời của Công ty là 14,4 triệu Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["S"], "claim_ir": {"metric": "renewable_energy", "value_text": "14,4", "unit_text": "Kwh", "multiplier_text": "triệu", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-C1", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C1 value", "claim": "Năng lượng mặt trời của Công ty là 18.026.961 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "18.026.961", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-C2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "C2 magnitude", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 nghìn Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["C"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "nghìn", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N2", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N2 status", "claim": "Công ty đặt mục tiêu năng lượng mặt trời đạt 14.421.569 Kwh vào năm 2030.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": 2030, "status": "target", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N5", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N5 basis", "claim": "Năng lượng mặt trời của Công ty là 14.421.569 Kwh/tấn sản phẩm.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh/tấn sản phẩm", "multiplier_text": "", "value_kind": "intensity", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n{"id": "bb-1d19e242d14134a1-0-N6", "layer": "B-b", "company": "vnm", "sector": "sữa", "family": "N6 metric", "claim": "Tổng lượng điện tiêu thụ của Công ty là 14.421.569 Kwh.", "evidence": "14.421.569 Kwh năng lượng mặt trời được sử dụng chiếm 8,49% tổng điện năng sử dụng (mảng sản xuất).", "labels": ["N"], "claim_ir": {"metric": "electricity_consumption", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "tổng lượng điện tiêu thụ", "subject_text": "", "extra": {}}, "evidence_ir": {"metric": "renewable_energy", "value_text": "14.421.569", "unit_text": "Kwh", "multiplier_text": "", "value_kind": "absolute", "change_direction": null, "baseline_year": null, "period": null, "status": "actual", "scope": null, "boundary": "unspecified", "metric_text": "năng lượng mặt trời", "subject_text": "", "extra": {}}, "fold": 0}\n',
    'jobs_b.json': '[\n {\n  "tag": "q3-8b",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-8B"\n   },\n   {\n    "repo": "Qwen/Qwen3-8B",\n    "backend": "hf"\n   }\n  ]\n },\n {\n  "tag": "q3-14b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-14B-AWQ"\n   },\n   {\n    "repo": "Qwen/Qwen3-14B-AWQ",\n    "backend": "hf"\n   }\n  ]\n },\n {\n  "tag": "gemma3-12b",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "unsloth/gemma-3-12b-it",\n    "kw": {\n     "limit_mm_per_prompt": {\n      "image": 0\n     }\n    }\n   },\n   {\n    "repo": "unsloth/gemma-3-12b-it",\n    "backend": "hf"\n   }\n  ]\n }\n]',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u multirun.py jobs_b.json ag1_worker.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u multirun.py jobs_b.json ag1_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)